# 0. les biblios

In [ ]:
import pandas as pd

import re

import matplotlib.pyplot as plt

import seaborn as sns

import numpy as np

# 1. Charger shipments.csv

In [ ]:
df = pd.read_csv("shipments.csv")

In [ ]:
print("Dimensions :", df.shape)

print("\nTypes de colonnes :")

df.info()

In [ ]:
print("\nPremières lignes :")

display(df.head())

- 452 lignes et 16 colonnes.

- Une ligne = une expédition (`shipment_number`) entre deux sites (`origin_facility_code` /

`destination_facility_code`), gérée par un transporteur (`carrier_partner_code`), avec un émetteur et un

destinataire (`sender_partner_code` / `receiver_partner_code`).

- Les colonnes de date incluent l'heure (`dispatch_time`, `arrival_time`) : plus précis que les fichiers

précédents qui n'avaient que le jour.

# 2. Doublons

### a- Doublons de lignes

In [ ]:
print("Lignes strictement dupliquées :", df.duplicated().sum())

display(df[df.duplicated(keep=False)].sort_values('shipment_number'))

### b- Doublons sur l'identifiant métier

In [ ]:
print("shipment_number dupliqués :", df['shipment_number'].duplicated().sum())

- Solution

In [ ]:
df = df.drop_duplicates(keep='first')

df = df.drop_duplicates(subset='shipment_number', keep='first')



print("Nombre de lignes après suppression :", df.shape[0])

print("Doublons restants :", df['shipment_number'].duplicated().sum())

### c- Doublons de colonnes

In [ ]:
print("Noms de colonnes en double :", df.columns[df.columns.duplicated()].tolist())

print("Colonnes au contenu identique :", df.columns[df.T.duplicated()].tolist())

print()

print(df['shipment_leg'].value_counts())

→ `shipment_leg` ne contient qu'une seule valeur (`1`), comme `purchase_order_line` dans les fichiers précédents. Conservée telle quelle.

# 3. Formats incohérents

In [ ]:
print(df.dtypes)

### - Vérifier le format des identifiants <--->

In [ ]:
print(df['shipment_number'].str.replace(r'\d', '9', regex=True).unique())

→ Format homogène.

### - Vérifier le format de dispatch_time et arrival_time (dates avec heure) <--->

In [ ]:
for col in ['dispatch_time', 'arrival_time']:

    dates_test = pd.to_datetime(df[col], errors='coerce')

    non_conv = dates_test.isna() & df[col].notna()

    print(col, "-> non convertibles :", non_conv.sum())

    display(df.loc[non_conv, col])

**Constat :** formats mixtes, comme sur `purchase_order_lines.csv` — la majorité au format ISO

(`2014-07-10T09:00:00`), mais quelques dates écrites `16/02/2018 12:00:00` ou `11-Aug-2019 18:00:00`.



**On applique directement la méthode en deux temps** (ISO strict d'abord, puis fallback `mixed+dayfirst`

seulement sur les échecs), pour éviter le piège rencontré sur le fichier précédent où `dayfirst=True`

appliqué à toute la colonne cassait les dates ISO déjà propres.

- Solution

In [ ]:
def parse_dates_mixtes(colonne):

    dates = pd.to_datetime(colonne, format='ISO8601', errors='coerce')

    echec = dates.isna() & colonne.notna()

    dates[echec] = pd.to_datetime(colonne[echec], format='mixed', dayfirst=True, errors='coerce')

    return dates



df['dispatch_time'] = parse_dates_mixtes(df['dispatch_time'])

df['arrival_time'] = parse_dates_mixtes(df['arrival_time'])



print(df[['dispatch_time', 'arrival_time']].dtypes)

print("Non convertibles (hors vides légitimes) :",

      (df['dispatch_time'].isna()).sum(), (df['arrival_time'].isna() & df['arrival_time'].notna()).sum())

print("Lignes où arrivée avant départ :", (df['arrival_time'] < df['dispatch_time']).sum())

### - Vérifier le format de declared_quantity (numérique)

In [ ]:
s = df['declared_quantity']

non_num = s[pd.to_numeric(s, errors='coerce').isna() & s.notna()]

print("Non convertibles :", len(non_num))

df['declared_quantity'] = pd.to_numeric(df['declared_quantity'], errors='coerce')

print(df['declared_quantity'].dtype)

→ Déjà propre, aucun problème de virgule ou d'unité recopiée ici.

### - Vérifier le format de temperature_control_class (catégorielle) <--->

In [ ]:
print(df['temperature_control_class'].value_counts(dropna=False))

**Constat :** casse incohérente — `2-8C`/`2-8c`, `ambient`/`Ambient`, `frozen`/`FROZEN` — 6 écritures

pour 3 vraies catégories.

- Solution

In [ ]:
df['temperature_control_class'] = df['temperature_control_class'].str.strip().str.lower()

print(df['temperature_control_class'].value_counts())

Remarque : `.lower()` transforme aussi `2-8C` en `2-8c` — c'est voulu, on garde une écriture

uniforme même si ce n'est pas une lettre de mot classique.

### - Vérifier le format de shipment_status (catégorielle) <--->

In [ ]:
print(df['shipment_status'].value_counts(dropna=False))

**Constat :** même souci de casse (`delivered`/`Delivered`/`DELIVERED`, `delayed`/`DELAYED`).

- Solution

In [ ]:
df['shipment_status'] = df['shipment_status'].str.strip().str.lower()

print(df['shipment_status'].value_counts())

### - Vérifier le format de declared_purpose (catégorielle) <--->

In [ ]:
print(df['declared_purpose'].value_counts(dropna=False))

**Constat, différent des fichiers précédents :** ce n'est pas un problème de casse, mais d'**espaces

multiples** — `"  laboratory  delivery "` au lieu de `"laboratory delivery"`. Un simple `.str.lower()`

n'aurait rien réglé ici.

- Solution

In [ ]:
df['declared_purpose'] = df['declared_purpose'].str.strip().str.replace(r'\s+', ' ', regex=True)



print(df['declared_purpose'].value_counts())

print("Nombre de catégories après nettoyage :", df['declared_purpose'].nunique())

`r'\s+'` capture un ou plusieurs espaces d'affilée et les remplace par un seul espace normal.

### - Vérifier le format de item_category, declared_unit, authorization_class

In [ ]:
for col in ['item_category', 'declared_unit', 'authorization_class']:

    print(col, "->", sorted(df[col].dropna().unique()))

    print()

→ Déjà propres, pas de souci de casse.

# 4. Valeurs manquantes

In [ ]:
print("Valeurs manquantes par colonne :")

print(df.isna().sum())

Seule `arrival_time` a des manquants (17 lignes, 3.8%). On vérifie s'il y a une logique cachée,

comme pour `weather_exception_code` ou `fault_code`.

In [ ]:
print(df[df['arrival_time'].isna()]['shipment_status'].value_counts())

**Constat confirmé :** les 17 lignes sans heure d'arrivée correspondent **exactement** aux 17 lignes

avec `shipment_status = in_transit` — un colis encore en transit n'a logiquement pas encore d'heure

d'arrivée. Ce n'est pas une perte de donnée, c'est une absence normale.



**Décision : on laisse `arrival_time` vide (`NaT`) pour ces lignes.** Contrairement à

`weather_exception_code` (une vraie catégorie), c'est une **date** — inventer une fausse heure d'arrivée

n'aurait aucun sens, et `NaT` représente déjà correctement "pas encore arrivé". On documente juste

l'explication.

# 5. Gestion des variables catégorielles

### a. Création de variables temporelles

In [ ]:
df['dispatch_year'] = df['dispatch_time'].dt.year

df['dispatch_month'] = df['dispatch_time'].dt.month

df['transit_hours'] = (df['arrival_time'] - df['dispatch_time']).dt.total_seconds() / 3600



display(df.head())

`transit_hours` est calculé uniquement pour les expéditions déjà arrivées (les 17 lignes `in_transit`

auront un `NaN` ici, ce qui est cohérent — on ne peut pas connaître une durée de transit pour un trajet

pas encore terminé).

### b. Variables ordonnées

`temperature_control_class` (`ambient` / `2-8c` / `frozen`) a une vraie notion de température

décroissante, mais ce n'est pas un ordre de "qualité" ou de "grandeur" au sens du cours — on la garde

nominale, comme `authorization_class` dans le fichier précédent.

### c. Encodage des variables nominales : reporté après le merge

Comme pour tous les fichiers logistiques, les catégories restent en texte nettoyé

(`item_category`, `declared_unit`, `temperature_control_class`, `authorization_class`, `declared_purpose`,

`shipment_status`). L'encodage One-Hot sera fait une seule fois, après le merge avec

`purchase_order_lines.csv`, `goods_receipts.csv`, `erp_facilities.csv`, `erp_business_partners.csv`.

In [ ]:
df_stats = df.copy()

print(df_stats.dtypes)

display(df_stats.head())

# 6. Voir les outliers (colonnes numériques)

In [ ]:
cols_num = ['declared_quantity', 'transit_hours']

display(df_stats[cols_num].describe().T)

In [ ]:
plt.figure(figsize=(10, 5))

for i, col in enumerate(cols_num, 1):

    plt.subplot(1, 2, i)

    sns.boxplot(y=df_stats[col], color="skyblue")

    plt.title(col)

plt.tight_layout()

plt.show()

In [ ]:
for col in cols_num:

    q1, q3 = df_stats[col].quantile([0.25, 0.75])

    iqr = q3 - q1

    n = ((df_stats[col] < q1 - 1.5*iqr) | (df_stats[col] > q3 + 1.5*iqr)).sum()

    print(f"{col:18s} min={df_stats[col].min():8.1f} max={df_stats[col].max():8.1f} -> {n} outliers")

**Constat, comme pour les autres fichiers logistiques :** `declared_quantity` mélange plusieurs

unités (`service`, `pallet`, `grant`...).

In [ ]:
display(df_stats.groupby('declared_unit')['declared_quantity'].mean().round(1))

Pour `transit_hours`, on vérifie si les valeurs extrêmes s'expliquent par un facteur logique

(distance, statut) :

In [ ]:
display(df_stats.groupby('shipment_status')['transit_hours'].describe()[['count', 'mean', 'min', 'max']])

→ Les expéditions `returned` ont un temps de transit nettement plus long (~62h en moyenne) que les

autres statuts. En revanche, `delayed` (~51h) et `delivered` (~49h) sont étonnamment proches — le retard

ne se voit pas forcément dans la durée totale de transit, il faudrait le comparer à une durée *attendue*

(pas disponible dans ce fichier) plutôt qu'à la durée des autres expéditions. **On conserve toutes les

valeurs**, aucune n'est impossible (pas de durée négative).

# 7. Histogrammes et distributions

In [ ]:
plt.figure(figsize=(10, 4))

for i, col in enumerate(cols_num, 1):

    plt.subplot(1, 2, i)

    sns.histplot(df_stats[col], kde=True, bins=25, color='skyblue')

    plt.title(f"Distribution : {col}")

plt.tight_layout()

plt.show()



print(df_stats[cols_num].skew().round(2))

In [ ]:
plt.figure(figsize=(14, 4))



plt.subplot(1, 3, 1)

sns.countplot(y='temperature_control_class', data=df_stats,

              order=df_stats['temperature_control_class'].value_counts().index, color='skyblue')

plt.title("Répartition par classe de température")



plt.subplot(1, 3, 2)

sns.countplot(y='shipment_status', data=df_stats,

              order=df_stats['shipment_status'].value_counts().index, color='skyblue')

plt.title("Répartition des statuts")



plt.subplot(1, 3, 3)

sns.countplot(y='declared_purpose', data=df_stats,

              order=df_stats['declared_purpose'].value_counts().index, color='skyblue')

plt.title("Répartition des motifs déclarés")



plt.tight_layout()

plt.show()

# 8. Corrélations

In [ ]:
matrice_spearman = df_stats[cols_num].corr(method='spearman')

matrice_pearson = df_stats[cols_num].corr(method='pearson')

display(matrice_spearman.round(2))

In [ ]:
plt.figure(figsize=(5, 4))

sns.heatmap(matrice_spearman, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)

plt.title("Matrice de corrélation de Spearman")

plt.tight_layout()

plt.show()

In [ ]:
sns.scatterplot(data=df_stats, x='declared_quantity', y='transit_hours', alpha=0.4, color='teal')

plt.title("Quantité déclarée vs durée de transit")

plt.show()

### - Analyse des corrélations



- `declared_quantity` et `transit_hours` ne sont quasiment pas corrélées : la durée de transit dépend

surtout de la distance/du transporteur, pas de la quantité transportée.

- Le lien le plus clair trouvé n'est pas numérique mais catégoriel : `transit_hours` varie fortement selon

`shipment_status` (vu à l'étape 6) — ce sera pertinent à recroiser avec les sites d'origine/destination

une fois fusionné avec `erp_facilities.csv` (distance géographique entre sites).

# 9. Export du jeu de données nettoyé

In [ ]:
df_stats.to_csv("shipments_clean.csv", index=False)



print("Dimensions finales :", df_stats.shape)

print("Valeurs manquantes restantes (hors arrival_time des colis en transit) :",

      df_stats.drop(columns=['arrival_time', 'transit_hours']).isna().sum().sum())

# 10. Synthèse du nettoyage



| Problème identifié | Colonnes concernées | Traitement appliqué |

|---|---|---|

| Doublons stricts (2) et doublons d'identifiant | toutes | suppression, on garde la 1ère occurrence |

| Dates/heures au format mixte | `dispatch_time`, `arrival_time` | parsing en 2 temps (ISO strict puis fallback `mixed+dayfirst`) — leçon tirée du fichier précédent |

| Casse incohérente | `temperature_control_class`, `shipment_status` | normalisation en minuscules |

| Espaces multiples (pas de casse) | `declared_purpose` | normalisation des espaces avec regex |

| 17 manquants expliqués par le statut | `arrival_time` | **laissés vides (NaT)** : confirmé que ce sont des colis encore `in_transit`, pas une perte de donnée |

| Encodage des catégorielles | toutes | **reporté** : fait une seule fois après le merge des fichiers logistiques |



**Biais introduits et assumés :**

- laisser `arrival_time` vide pour les colis en transit suppose que ces expéditions seront un jour

complétées dans une future mise à jour du fichier — sinon elles resteront durablement incomplètes dans

toute analyse de durée de transit ;

- comme pour les fichiers précédents, on garde la 1ère occurrence des doublons après vérification qu'ils

étaient identiques.